In [3]:
pip install gradio

  Using cached gradio-6.14.0-py3-none-any.whl.metadata (17 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached fastapi-0.136.1-py3-none-any.whl.metadata (28 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached huggingface_hub-1.14.0-py3-none-any.whl.metadata (14 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.27-py3-none-any.whl.metadata (2.1 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9

In [ ]:
import gradio as gr
import numpy as np
import librosa
import tensorflow as tf
import os

# ── Parámetros ────────────────────────────────────────────────────────────────
TARGET_SR   = 16000
FRAME_LEN   = 2048
N_FRAMES    = 5
RUTA_MODELO = '../Metricas/modelo2_1_balanceado.keras'

modelo = tf.keras.models.load_model(RUTA_MODELO)

def calcular_segmentos_uniformes(total_samples):
    segment_size = total_samples // N_FRAMES
    indices = []
    for i in range(N_FRAMES):
        center = (i * segment_size) + (segment_size // 2)
        start  = center - (FRAME_LEN // 2)
        end    = start + FRAME_LEN
        if start < 0:           start, end = 0, FRAME_LEN
        if end > total_samples: start, end = total_samples - FRAME_LEN, total_samples
        indices.append((start, end))
    return indices

def clasificar_audio(audio_path):
    audio, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)

    frames = calcular_segmentos_uniformes(len(audio))
    features = []
    for start, end in frames:
        stft = librosa.stft(audio[start:end], n_fft=FRAME_LEN, hop_length=FRAME_LEN + 1)
        features.append(np.abs(stft))

    tensor = np.expand_dims(np.array(features).squeeze(), axis=0)
    prob   = modelo.predict(tensor, verbose=0)[0][0]

    if prob > 0.5:
        etiqueta = f"🔴 SPOOF — voz sintética detectada"
        color    = "tomato"
    else:
        etiqueta = f"🟢 BONAFIDE — voz humana real"
        color    = "seagreen"

    resultado = f"""
    <div style='text-align:center; padding:20px; border-radius:12px;
                background-color:{color}; color:white; font-size:22px; font-weight:bold;'>
        {etiqueta}
    </div>
    <div style='text-align:center; padding:12px; font-size:16px; margin-top:10px;'>
        P(spoof) = <b>{prob:.3f}</b> &nbsp;|&nbsp; P(bonafide) = <b>{(1-prob):.3f}</b>
    </div>
    """
    return resultado

demo = gr.Interface(
    fn=clasificar_audio,
    inputs=gr.Audio(type="filepath", label="Sube o graba un audio"),
    outputs=gr.HTML(label="Resultado"),
    title="🎙️ Detector de Voz Sintética",
    description="Modelo CNN entrenado sobre ASVspoof 2019 + dataset latinoamericano. "
                "Sube un audio en formato WAV o M4A y el modelo determinará si es voz humana real (bonafide) o sintética (spoof).",
    examples=[
        ["../test_voces/Liliana_VozEspañolNotaAudioIphone_16k.wav"],
        ["../test_voces/Daniele_VozItaNotaAudioIphone_16k.wav"],
        ["../test_voces/Lian_CancionNotaAudioIphone_16k.wav"],
    ],
    theme=gr.themes.Soft()
)

demo.launch()


/Users/liliana/venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/liliana/venv312/lib/python3.12/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/liliana/venv312/lib/python3.12/site-packages/gradio/routes.py", line 1474, in predict
    output = await route_utils.call_process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/liliana/venv312/lib/python3.12/site-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/liliana/venv312/lib/python3.12/site-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/liliana/venv312/lib/python3.12/site-packages/gradio/blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/liliana/venv312/lib/python3.12/site-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backe